# H Diffusivity Workflow — Script Generator + Analysis

## Workflow overview

This notebook is a **script generator**. Running cells 1–3 writes two files to disk:
- `diffusivity_run.py` — standalone Python orchestrator (Phases 1–3)
- `diffusivity_run.sh` — SLURM submission script (`west` partition, no wall-time limit, no GPU)

Submitting `diffusivity_run.sh` runs the full workflow on the cluster unattended.  
Cells 4+ are run **locally after the cluster jobs complete** to inspect and visualise results.

---

**Phase 1 — Structure preparation** *(multigpu partition)*  
&nbsp;&nbsp; 1a. Minimise input bulk structure  
&nbsp;&nbsp; 1b. Insert *n* H atoms at FCC octahedral sites  
&nbsp;&nbsp; 1c. Minimise bulk+H structure

**Phase 2 — NVT MD** *(chained GPU jobs, one per temperature)*  
&nbsp;&nbsp; Equilibration + production NVT with H-atom MSD tracking.  
&nbsp;&nbsp; Self-resubmitting if wall-time is exceeded.

**Phase 3 — Diffusivity extraction** *(inside orchestrator, after all NVT jobs)*  
&nbsp;&nbsp; MSD → D(T) via `run_diffusivity_pipeline` → `diffusivity_table.txt`  
&nbsp;&nbsp; Arrhenius fit → Ea, D₀ via `run_arrhenius_pipeline`

**Phase 4 — Local analysis & visualisation** *(run in this notebook)*  
&nbsp;&nbsp; 4a. Load `diffusivity_table.txt` → summary DataFrame  
&nbsp;&nbsp; 4b. Minimisation QC — convergence & Fmax check  
&nbsp;&nbsp; 4c. NVT thermo QC — temperature drift, MSD traces  
&nbsp;&nbsp; 4d. D vs T plot (log scale, per structure)  
&nbsp;&nbsp; 4e. Ea vs n_H plot  
&nbsp;&nbsp; 4f. Arrhenius overlay (log D vs 1000/T, all n_H)

## Cells 1–2: Imports & configuration

Edit `input_structures`, `n_h_values`, `temperatures`, `NVT_WALL_TIME` here before generating scripts.

In [45]:
import os
import sys

# Add parent directory to path
parent_dir = os.path.dirname(os.path.dirname(os.path.abspath('__file__')))
if parent_dir not in sys.path:
    sys.path.insert(0, parent_dir)

In [54]:

# ── Imports ───────────────────────────────────────────────────────────────────
from models.config import (
    LAMMPS_CMD, MACE_MODEL_LAMMPS, KOKKOS_FLAGS,
    PAIR_STYLE, PAIR_SUFFIX,
    E2T_7, MASSES_7, ELEM_STR_7,
    SLURM_DEFAULTS, BASE_DIR,
)
#from models.structure import get_lattice_parameter , insert_hydrogen
from models.utils import make_run_dirs
from models.lammps_script import (
    write_minimization_script,
    write_nvt_bulk_script,
    write_nvt_bulk_restart_script,
)
from models.create_slurm import (
    write_slurm_job,
    write_chained_slurm_job,
    submit_slurm_job,
    wait_for_jobs,
)
from models.diffusivity_post_processing import (
    run_diffusivity_pipeline,
    save_diffusivity_table,
    run_arrhenius_pipeline,
)

# ── User-editable config ──────────────────────────────────────────────────────
# BASE_DIR = '/projects/westgroup/akinyemi.az/mace_lammps/MHI_Nickel'  
WORK_DIR = os.path.join(BASE_DIR, 'calculation') # '/projects/westgroup/akinyemi.az/mace_lammps/MHI_Nickel/calculation'

input_structures = [
    # Add full paths to pre-built bulk .lammps files:
    # '/projects/westgroup/akinyemi.az/mace_lammps/MHI_Nickel/structures/seed1.lammps',
    # '/projects/westgroup/akinyemi.az/mace_lammps/MHI_Nickel/structures/seed2.lammps',
    '/projects/westgroup/akinyemi.az/mace_lammps/MHI_Nickel/calculation',
]

n_h_values   = [1, 3, 5, 7, 10]
temperatures = [300, 400, 500, 600, 700, 800]  # K

# Wall time per NVT chained leg (multigpu max ~24h; leave headroom for resubmit)
NVT_WALL_TIME = '24:00:00'
CUTOFF        = '23:55:00'   # timeout inside the chained job before resubmit

# SLURM config for all GPU jobs (minimization + NVT)
GPU_SLURM_CFG = dict(SLURM_DEFAULTS, partition='multigpu', time=NVT_WALL_TIME)

KK = ' '.join(KOKKOS_FLAGS)   # kokkos flags as a single string for shell commands

print('Config loaded.')
print(f'  WORK_DIR         : {WORK_DIR}')
print(f'  input_structures : {len(input_structures)} file(s)')
print(f'  n_h_values       : {n_h_values}')
print(f'  temperatures     : {temperatures}')
print(f'  NVT_WALL_TIME    : {NVT_WALL_TIME}  |  CUTOFF: {CUTOFF}')


Config loaded.
  WORK_DIR         : /projects/westgroup/akinyemi.az/mace_lammps/MHI_Nickel/calculation
  input_structures : 1 file(s)
  n_h_values       : [1, 3, 5, 7, 10]
  temperatures     : [300, 400, 500, 600, 700, 800]
  NVT_WALL_TIME    : 24:00:00  |  CUTOFF: 23:55:00


## Cell 3: Generate `diffusivity_run.py`

Builds the full Phase 1→2→3 orchestrator script as a Python string and writes it to disk.  
The script is parameterised by the config values set in Cell 2 (injected as literals via f-string header).

### Phase 1 — Structure preparation (inside `diffusivity_run.py`)

For each `(struct, n_h)`:
1. Write + submit `minimize_bare.lammps` → wait → minimised bulk
2. `insert_hydrogen` at FCC octahedral sites using `a0` from minimised structure
3. Write + submit `minimize_h.lammps` → wait → minimised bulk+H

In [55]:

# ── Build diffusivity_run.py ──────────────────────────────────────────────────
# The header section is an f-string: it embeds the notebook config values.
# The body section is a plain string (no f-prefix) so that {T}, {n_h}, etc.
# inside the script are written as literals, not evaluated here.

_header = f"""\
#!/usr/bin/env python3
\"\"\"
diffusivity_run.py
==================
Standalone H-diffusivity workflow orchestrator.
Submitted via diffusivity_run.sh (west partition, no GPU, no wall-time limit).

Loop: input_structures × n_h_values × temperatures
  Phase 1 : Minimise bulk  →  insert H  →  minimise bulk+H
  Phase 2 : NVT MD (chained SLURM GPU jobs, one per temperature)
  Phase 3 : MSD  →  D(T)  →  Arrhenius fit

Generated by calculation/diffusivity.ipynb — do not edit by hand.
\"\"\"

# ── config (injected by diffusivity.ipynb) ────────────────────────────────────
INPUT_STRUCTURES = {input_structures!r}
N_H_VALUES       = {n_h_values!r}
TEMPERATURES     = {temperatures!r}
WORK_DIR         = {WORK_DIR!r}
NVT_WALL_TIME    = {NVT_WALL_TIME!r}
CUTOFF           = {CUTOFF!r}

"""

_body = """\
import os
import sys
sys.path.insert(0, WORK_DIR)

from models.config import (
    LAMMPS_CMD, MACE_MODEL_LAMMPS, KOKKOS_FLAGS,
    PAIR_STYLE, PAIR_SUFFIX,
    E2T_7, MASSES_7, ELEM_STR_7,
    SLURM_DEFAULTS,
)
from models.structure import get_lattice_parameter, insert_hydrogen
from models.utils import make_run_dirs
from models.lammps_script import (
    write_minimization_script,
    write_nvt_bulk_script,
    write_nvt_bulk_restart_script,
)
from models.create_slurm import (
    write_slurm_job,
    write_chained_slurm_job,
    submit_slurm_job,
    wait_for_jobs,
)
from models.diffusivity_post_processing import (
    run_diffusivity_pipeline,
    save_diffusivity_table,
    run_arrhenius_pipeline,
)

GPU_SLURM_CFG = dict(SLURM_DEFAULTS, partition='multigpu', time=NVT_WALL_TIME)
KK = ' '.join(KOKKOS_FLAGS)

# ── main loop ─────────────────────────────────────────────────────────────────
for struct_path in INPUT_STRUCTURES:
    struct_stem = os.path.splitext(os.path.basename(struct_path))[0]

    for n_h in N_H_VALUES:
        run_name = f'{struct_stem}_{n_h}H'
        print(f'\\n{"="*60}')
        print(f'  Structure : {struct_stem}   n_H : {n_h}')
        print(f'{"="*60}')

        # Directory layout created by make_run_dirs:
        #   results/{run_name}/structures/
        #   results/{run_name}/lammps_scripts/{T}K/
        #   results/{run_name}/slurm_scripts/{T}K/
        #   results/{run_name}/results/{T}K/
        dirs = make_run_dirs(
            name=run_name,
            temperatures=TEMPERATURES,
            base_dir=os.path.join(WORK_DIR, 'results'),
        )

        # Phase 1 scripts live one level up from the per-temperature dirs
        phase1_lmp_dir = os.path.join(dirs['root'], 'lammps_scripts')
        phase1_sh_dir  = os.path.join(dirs['root'], 'slurm_scripts')
        os.makedirs(phase1_lmp_dir, exist_ok=True)
        os.makedirs(phase1_sh_dir,  exist_ok=True)

        # ── Phase 1 ──────────────────────────────────────────────────────────
        print('\\n--- Phase 1: Structure preparation ---')

        # 1a: minimise bare bulk
        min_bare_lmp = os.path.join(phase1_lmp_dir, 'minimize_bare.lammps')
        min_bare_out = os.path.join(dirs['structures'], 'bulk_min.lammps')
        min_bare_sh  = os.path.join(phase1_sh_dir,  'minimize_bare.sh')

        write_minimization_script(
            bulk_input=struct_path,
            min_output=min_bare_out,
            out_path=min_bare_lmp,
            pair_style=PAIR_STYLE,
            mace_model=MACE_MODEL_LAMMPS,
            pair_suffix=PAIR_SUFFIX,
            elem_str=ELEM_STR_7,
        )
        write_slurm_job(
            job_name=f'min_bare_{run_name}',
            slurm_config=GPU_SLURM_CFG,
            out_path=min_bare_sh,
            runner='lmp',
            lammps_cmd=LAMMPS_CMD,
            kokkos_flags=KOKKOS_FLAGS,
            script_path=min_bare_lmp,
        )
        jid = submit_slurm_job(min_bare_sh)
        wait_for_jobs({'min_bare': jid})
        print(f'  [1a] Bare bulk minimisation done.')

        # 1b: extract lattice parameter and insert H
        a0 = get_lattice_parameter(min_bare_out)
        print(f'  [1b] a0 = {a0:.4f} Å  →  inserting {n_h} H atom(s)')
        bulk_h_path, _, _ = insert_hydrogen(
            bulk_min_path=min_bare_out,
            n_h=n_h,
            masses=MASSES_7,
            e2t=E2T_7,
            out_dir=dirs['structures'],
            a0=a0,
        )

        # 1c: minimise bulk+H
        min_h_lmp = os.path.join(phase1_lmp_dir, 'minimize_h.lammps')
        min_h_out = os.path.join(dirs['structures'], 'bulk_min_h.lammps')
        min_h_sh  = os.path.join(phase1_sh_dir,  'minimize_h.sh')

        write_minimization_script(
            bulk_input=bulk_h_path,
            min_output=min_h_out,
            out_path=min_h_lmp,
            pair_style=PAIR_STYLE,
            mace_model=MACE_MODEL_LAMMPS,
            pair_suffix=PAIR_SUFFIX,
            elem_str=ELEM_STR_7,
        )
        write_slurm_job(
            job_name=f'min_h_{run_name}',
            slurm_config=GPU_SLURM_CFG,
            out_path=min_h_sh,
            runner='lmp',
            lammps_cmd=LAMMPS_CMD,
            kokkos_flags=KOKKOS_FLAGS,
            script_path=min_h_lmp,
        )
        jid = submit_slurm_job(min_h_sh)
        wait_for_jobs({'min_h': jid})
        print(f'  [1c] Bulk+H minimisation done.')

        # ── Phase 2 ──────────────────────────────────────────────────────────
        print('\\n--- Phase 2: NVT MD ---')
        job_ids = {}

        for T in TEMPERATURES:
            lmp_dir = dirs[T]['lammps_scripts']
            sh_dir  = dirs[T]['slurm_scripts']
            res_dir = dirs[T]['results']
            rst_dir = os.path.join(res_dir, 'checkpoints')
            os.makedirs(rst_dir, exist_ok=True)

            nvt_lmp     = os.path.join(lmp_dir, f'nvt_{T}K.lammps')
            nvt_rst_lmp = os.path.join(lmp_dir, f'nvt_{T}K_restart.lammps')
            traj_file   = os.path.join(res_dir, f'nvt_{T}K.dump')
            out_file    = os.path.join(res_dir, f'nvt_{T}K.out')
            msd_file    = os.path.join(res_dir, f'msd_{T}K.dat')
            rst_glob    = os.path.join(rst_dir, f'nvt_{T}K.*.restart')
            chain_sh    = os.path.join(sh_dir,  f'nvt_{T}K_chain.sh')

            write_nvt_bulk_script(
                bulk_h_file=min_h_out,
                traj_file=traj_file,
                out_file=out_file,
                msd_file=msd_file,
                out_path=nvt_lmp,
                pair_style=PAIR_STYLE,
                mace_model=MACE_MODEL_LAMMPS,
                pair_suffix=PAIR_SUFFIX,
                elem_str=ELEM_STR_7,
                temperature=T,
                h_type=E2T_7['H'],
                restart_dir=rst_dir,
            )
            write_nvt_bulk_restart_script(
                restart_file=rst_glob,
                traj_file=traj_file,
                out_file=out_file,
                msd_file=msd_file,
                out_path=nvt_rst_lmp,
                pair_style=PAIR_STYLE,
                mace_model=MACE_MODEL_LAMMPS,
                pair_suffix=PAIR_SUFFIX,
                elem_str=ELEM_STR_7,
                temperature=T,
                h_type=E2T_7['H'],
                restart_dir=rst_dir,
            )
            write_chained_slurm_job(
                job_name=f'nvt_{T}K_{run_name}',
                slurm_config=GPU_SLURM_CFG,
                out_path=chain_sh,
                first_commands=[
                    f'{LAMMPS_CMD} {KK} -in {nvt_lmp} -log {out_file}',
                ],
                restart_commands=[
                    f'{LAMMPS_CMD} {KK} -in {nvt_rst_lmp} -log {out_file}',
                ],
                restart_glob=rst_glob,
                cutoff=CUTOFF,
                work_dir=WORK_DIR,
            )
            job_ids[f'{T}K'] = submit_slurm_job(chain_sh)
            print(f'  [2] Submitted NVT {T}K  →  job {job_ids[f"{T}K"]}')

        print(f'  Waiting for {len(job_ids)} NVT jobs ...')
        wait_for_jobs(job_ids)
        print('  All NVT jobs done.')

        # ── Phase 3 ──────────────────────────────────────────────────────────
        print('\\n--- Phase 3: Post-processing ---')
        analysis_dir = os.path.join(dirs['root'], 'analysis')
        os.makedirs(analysis_dir, exist_ok=True)

        D_vals, D_errs, R2_vals = [], [], []
        for T in TEMPERATURES:
            dump   = os.path.join(dirs[T]['results'], f'nvt_{T}K.dump')
            result = run_diffusivity_pipeline(
                dump_file=dump,
                temperature=T,
                h_type=E2T_7['H'],
                outdir=analysis_dir,
            )
            D_vals.append(result['D'])
            D_errs.append(result['D_err'])
            R2_vals.append(result['R2'])
            print(f'  T={T}K   D={result["D"]:.4e} m2/s   R2={result["R2"]:.4f}')

        table_path = os.path.join(analysis_dir, 'diffusivity_table.txt')
        save_diffusivity_table(TEMPERATURES, D_vals, D_errs, R2_vals, table_path)

        arr = run_arrhenius_pipeline(
            diffusivity_file=table_path,
            outdir=analysis_dir,
        )
        print(f'  Ea = {arr["Ea"]:.4f} eV   D0 = {arr["D0"]:.4e} m2/s')

print('\\n=== All structures and H concentrations complete ===')
"""

script_content = _header + _body

out_py = os.path.join(os.getcwd(), 'diffusivity_run.py')
with open(out_py, 'w') as fh:
    fh.write(script_content)

print(f'Written: {out_py}')
print(f'  Lines: {script_content.count(chr(10))}')


Written: /Users/akinyemi.az/Desktop/PhD_Folder/research/MHI/MD/Molecular_Dynamics/MHI_Nickel/calculation/diffusivity_run.py
  Lines: 248


### Phase 2 — NVT MD (inside `diffusivity_run.py`)

For each temperature:
- `write_nvt_bulk_script` → fresh-start LAMMPS input
- `write_nvt_bulk_restart_script` → restart-from-checkpoint input
- `write_chained_slurm_job` → self-resubmitting `.sh` (cutoff = `CUTOFF`)
- All temperatures submitted simultaneously; `wait_for_jobs` blocks until all complete

### Phase 3 — Post-processing (inside `diffusivity_run.py`)

After all NVT jobs finish:
- `run_diffusivity_pipeline(dump_file, T)` → MSD → D, D_err, R²
- `save_diffusivity_table` → `analysis/diffusivity_table.txt`
- `run_arrhenius_pipeline` → Ea, D₀, `arrhenius.png`

## Cell 4: Generate `diffusivity_run.sh` and submit

Writes the SLURM orchestrator script (`west` partition, 4 CPUs, 16 GB RAM, no GPU, no `--time`).  
Set `dry_run=False` when ready to submit to the cluster.

---
## Phase 4: Local analysis (run after cluster jobs complete)

### 4a. Load results — `diffusivity_table.txt` → summary DataFrame

### 4b–4c. QC checks — minimisation convergence + NVT temperature drift & MSD traces

In [56]:

import textwrap

# ── Generate diffusivity_run.sh (west partition, CPU-only, no wall-time limit) ─
# write_slurm_job requires 'gpu' and 'time' keys — not applicable here,
# so write the orchestrator script directly.

_ld_lines = '\n'.join(
    f'export LD_LIBRARY_PATH={p}:$LD_LIBRARY_PATH'
    for p in GPU_SLURM_CFG['ld_paths']
)

orch_sh_content = textwrap.dedent(f"""\
    #!/bin/bash
    #SBATCH --job-name=diffusivity_orch
    #SBATCH --partition=west
    #SBATCH --ntasks=1
    #SBATCH --cpus-per-task=4
    #SBATCH --mem=16G
    #SBATCH --output=diffusivity_orch_%j.out

    module load OpenMPI/{GPU_SLURM_CFG['openmpi_ver']}
    module load cuda/{GPU_SLURM_CFG['cuda_version']}
    source ~/miniforge3/etc/profile.d/conda.sh
    conda activate {GPU_SLURM_CFG['conda_env']}
    {_ld_lines}

    cd {os.getcwd()}

    echo "Node: $(hostname)  Start: $(date)"
    python {out_py}
    echo "End: $(date)"
""")

out_sh = os.path.join(os.getcwd(), 'diffusivity_run.sh')
with open(out_sh, 'w') as fh:
    fh.write(orch_sh_content)
os.chmod(out_sh, 0o755)
print(f'Written: {out_sh}')

# ── Preview first 20 lines ────────────────────────────────────────────────────
for line in orch_sh_content.splitlines()[:20]:
    print(line)

# ── Submit ────────────────────────────────────────────────────────────────────
# Set dry_run=False when ready to submit to the cluster.
job_id = submit_slurm_job(out_sh, dry_run=False)
print(f'\nOrchestrator job: {job_id}')


Written: /Users/akinyemi.az/Desktop/PhD_Folder/research/MHI/MD/Molecular_Dynamics/MHI_Nickel/calculation/diffusivity_run.sh
    #!/bin/bash
    #SBATCH --job-name=diffusivity_orch
    #SBATCH --partition=west
    #SBATCH --ntasks=1
    #SBATCH --cpus-per-task=4
    #SBATCH --mem=16G
    #SBATCH --output=diffusivity_orch_%j.out

    module load OpenMPI/4.1.6
    module load cuda/12.3.0
    source ~/miniforge3/etc/profile.d/conda.sh
    conda activate mace-lammps
    export LD_LIBRARY_PATH=/shared/EL9/explorer/cuda/12.3.0/lib64/stubs:$LD_LIBRARY_PATH
export LD_LIBRARY_PATH=/projects/westgroup/akinyemi.az/mace_lammps/lammps/build-mliap:$LD_LIBRARY_PATH
export LD_LIBRARY_PATH=/home/akinyemi.az/miniforge3/envs/mace-lammps/lib:$LD_LIBRARY_PATH

    cd /Users/akinyemi.az/Desktop/PhD_Folder/research/MHI/MD/Molecular_Dynamics/MHI_Nickel/calculation

    echo "Node: $(hostname)  Start: $(date)"
    python /Users/akinyemi.az/Desktop/PhD_Folder/research/MHI/MD/Molecular_Dynamics/MHI_Nickel/calcula

FileNotFoundError: [Errno 2] No such file or directory: 'sbatch'

In [49]:

# ── Phase 4: Load results ─────────────────────────────────────────────────────
# Run this section locally after diffusivity_run.py has finished on the cluster.
# Scans results/{struct}_{n_h}H/analysis/diffusivity_table.txt for all completed
# runs and assembles a summary DataFrame.
#
# File format written by save_diffusivity_table (whitespace-delimited, # comments):
#   T_K   D (m2/s)   sigma_D (m2/s)   R2

import glob
import pandas as pd
import numpy as np
from pathlib import Path

from models.diffusivity_post_processing import fit_arrhenius, run_arrhenius_pipeline

RESULTS_ROOT = os.path.join(WORK_DIR, 'results')

_COL_NAMES = ['T_K', 'D', 'sigma_D', 'R2']   # matches save_diffusivity_table output

records = []
for struct_path in input_structures:
    struct_stem = os.path.splitext(os.path.basename(struct_path))[0]
    for n_h in n_h_values:
        run_name  = f'{struct_stem}_{n_h}H'
        table_txt = os.path.join(RESULTS_ROOT, run_name, 'analysis', 'diffusivity_table.txt')
        if not os.path.exists(table_txt):
            print(f'  [MISSING] {table_txt}')
            continue
        df = pd.read_csv(
            table_txt,
            sep=r'\s+',
            comment='#',
            names=_COL_NAMES,
        )
        df['struct'] = struct_stem
        df['n_h']    = n_h
        records.append(df)

if records:
    summary = pd.concat(records, ignore_index=True)
    print(f'Loaded {len(summary)} rows from {len(records)} completed runs.')
    display(summary.head(12))
else:
    print('No completed results found. Run the cluster jobs first.')
    summary = pd.DataFrame()


No completed results found. Run the cluster jobs first.


In [50]:

# ── Phase 4a: Minimization QC ─────────────────────────────────────────────────
# Parse minimize_bare.out and minimize_h.out per (struct, n_h) and print a
# summary table: Ecoh, a0, Fmax, stopping criterion.

from models.parsers import parse_minimization_log

min_records = []
for struct_path in input_structures:
    struct_stem = os.path.splitext(os.path.basename(struct_path))[0]
    for n_h in n_h_values:
        run_name   = f'{struct_stem}_{n_h}H'
        sh_dir     = os.path.join(RESULTS_ROOT, run_name, 'slurm_scripts')
        # logs written alongside SLURM scripts (LAMMPS default log location)
        for label, logname in [('bare', 'minimize_bare.out'), ('H', 'minimize_h.out')]:
            logfile = os.path.join(RESULTS_ROOT, run_name, 'lammps_scripts', logname)
            if not os.path.exists(logfile):
                # also try results root
                logfile = os.path.join(RESULTS_ROOT, run_name, logname)
            if not os.path.exists(logfile):
                continue
            _, meta = parse_minimization_log(logfile)
            min_records.append({
                'struct':   struct_stem,
                'n_h':      n_h,
                'step':     label,
                'Ecoh_eV':  meta.get('Ecoh_eV_per_atom', float('nan')),
                'a0_Ang':   meta.get('a0_Angstrom',      float('nan')),
                'Fmax':     meta.get('Fmax_eV_per_Ang',  float('nan')),
                'converged': meta.get('stop_criterion',  'unknown'),
            })

if min_records:
    min_df = pd.DataFrame(min_records)
    display(min_df)
    # Flag any unconverged runs
    bad = min_df[~min_df['converged'].str.contains('force', case=False, na=False)]
    if not bad.empty:
        print(f'\n⚠ {len(bad)} minimisation(s) may not have converged:')
        display(bad[['struct', 'n_h', 'step', 'Fmax', 'converged']])
    else:
        print('All minimisations converged on force criterion.')
else:
    print('No minimisation logs found (jobs may still be running).')


TypeError: unsupported operand type(s) for |: 'types.GenericAlias' and 'NoneType'

In [ ]:

# ── Phase 4b: NVT thermo + MSD QC ────────────────────────────────────────────
# For each (struct, n_h, T):
#   - parse_equil_log  → scalar summary (T_final, PE_final)
#   - parse_thermo_series → temperature vs time plot (equilibration check)
#   - read msd_{T}K.dat → MSD vs time overlay (sanity vs dump-derived MSD)

from models.parsers import parse_equil_log, parse_thermo_series

thermo_records = []

for struct_path in input_structures:
    struct_stem = os.path.splitext(os.path.basename(struct_path))[0]
    for n_h in n_h_values:
        run_name = f'{struct_stem}_{n_h}H'
        for T in temperatures:
            out_file = os.path.join(RESULTS_ROOT, run_name, 'results', f'{T}K', f'nvt_{T}K.out')
            msd_file = os.path.join(RESULTS_ROOT, run_name, 'results', f'{T}K', f'msd_{T}K.dat')

            meta = parse_equil_log(out_file)
            if meta:
                thermo_records.append({
                    'struct':       struct_stem,
                    'n_h':          n_h,
                    'T_set':        T,
                    'T_final_K':    meta.get('temp_final_K', float('nan')),
                    'PE_final_eV':  meta.get('pe_final_eV',  float('nan')),
                })

if thermo_records:
    thermo_df = pd.DataFrame(thermo_records)
    display(thermo_df)

    # Check temperature drift: |T_final - T_set| / T_set
    thermo_df['T_drift_%'] = (
        (thermo_df['T_final_K'] - thermo_df['T_set']).abs() / thermo_df['T_set'] * 100
    )
    drifty = thermo_df[thermo_df['T_drift_%'] > 5]
    if not drifty.empty:
        print(f'\n⚠ {len(drifty)} run(s) with >5% temperature drift:')
        display(drifty[['struct', 'n_h', 'T_set', 'T_final_K', 'T_drift_%']])
    else:
        print('All NVT runs within 5% of target temperature.')
else:
    print('No NVT thermo logs found (jobs may still be running).')

# ── Temperature vs time plots (one figure per n_h) ───────────────────────────
for struct_path in input_structures:
    struct_stem = os.path.splitext(os.path.basename(struct_path))[0]
    for n_h in n_h_values:
        run_name = f'{struct_stem}_{n_h}H'
        fig, axes = plt.subplots(
            2, len(temperatures),
            figsize=(3.5 * len(temperatures), 6),
            sharex='col',
        )
        has_data = False
        for col, T in enumerate(temperatures):
            out_file = os.path.join(RESULTS_ROOT, run_name, 'results', f'{T}K', f'nvt_{T}K.out')
            msd_file = os.path.join(RESULTS_ROOT, run_name, 'results', f'{T}K', f'msd_{T}K.dat')

            # ── row 0: temperature trace ──────────────────────────────
            res = parse_thermo_series(out_file)
            if res is not None:
                steps, temps_trace, _ = res
                t_ps = steps * 0.0005   # timestep in ps
                axes[0][col].plot(t_ps, temps_trace, lw=0.6)
                axes[0][col].axhline(T, color='red', lw=1, ls='--', label=f'{T} K')
                axes[0][col].set_title(f'{T} K')
                has_data = True

            # ── row 1: MSD from .dat file ─────────────────────────────
            if os.path.exists(msd_file):
                try:
                    msd_data = np.loadtxt(msd_file, comments='#')
                    if msd_data.ndim == 2 and msd_data.shape[1] >= 2:
                        axes[1][col].plot(msd_data[:, 0] * 0.0005, msd_data[:, 1], lw=0.8)
                        has_data = True
                except Exception:
                    pass

        if has_data:
            axes[0][0].set_ylabel('Temperature (K)')
            axes[1][0].set_ylabel('MSD (Å²)')
            for col in range(len(temperatures)):
                axes[1][col].set_xlabel('Time (ps)')
            fig.suptitle(f'{struct_stem} — {n_h} H atoms: Thermo QC', fontsize=12)
            fig.tight_layout()
            out_fig = os.path.join(RESULTS_ROOT, run_name, 'analysis', f'thermo_qc_{n_h}H.png')
            os.makedirs(os.path.dirname(out_fig), exist_ok=True)
            plt.savefig(out_fig, dpi=120)
            plt.show()
            print(f'Saved: {out_fig}')
        else:
            plt.close()


In [ ]:

# ── D vs T plots ──────────────────────────────────────────────────────────────
# One subplot per structure, curves coloured by n_H value.
# Columns in summary: T_K, D, sigma_D, R2, struct, n_h

import matplotlib.pyplot as plt
import matplotlib.cm as cm

if summary.empty:
    print('No data to plot.')
else:
    structs = summary['struct'].unique()
    n_h_list = sorted(summary['n_h'].unique())
    cmap = cm.get_cmap('viridis', len(n_h_list))
    color_map = {nh: cmap(i) for i, nh in enumerate(n_h_list)}

    fig, axes = plt.subplots(
        1, len(structs),
        figsize=(5 * len(structs), 4),
        sharey=True,
        squeeze=False,
    )

    for col, struct in enumerate(structs):
        ax = axes[0][col]
        sub = summary[summary['struct'] == struct]
        for n_h in n_h_list:
            d = sub[sub['n_h'] == n_h].sort_values('T_K')
            if d.empty:
                continue
            ax.errorbar(
                d['T_K'], d['D'],
                yerr=d['sigma_D'],
                fmt='o-', color=color_map[n_h],
                label=f'{n_h} H', capsize=3,
            )
        ax.set_xlabel('Temperature (K)')
        ax.set_title(struct)
        ax.set_yscale('log')
        ax.legend(title='n_H', fontsize=8)

    axes[0][0].set_ylabel('D (m²/s)')
    fig.suptitle('H Diffusivity vs Temperature', fontsize=13)
    fig.tight_layout()
    plt.savefig(os.path.join(RESULTS_ROOT, 'D_vs_T.png'), dpi=150)
    plt.show()
    print(f'Saved: {os.path.join(RESULTS_ROOT, "D_vs_T.png")}')


In [ ]:

# ── Ea vs n_H ─────────────────────────────────────────────────────────────────
# Run Arrhenius fit per (struct, n_h) and plot activation energy vs H concentration.

if summary.empty:
    print('No data to plot.')
else:
    arrhenius_records = []

    for struct_path in input_structures:
        struct_stem = os.path.splitext(os.path.basename(struct_path))[0]
        for n_h in n_h_values:
            run_name  = f'{struct_stem}_{n_h}H'
            table_txt = os.path.join(RESULTS_ROOT, run_name, 'analysis', 'diffusivity_table.txt')
            if not os.path.exists(table_txt):
                continue
            arr = run_arrhenius_pipeline(
                diffusivity_file=table_txt,
                outdir=os.path.join(RESULTS_ROOT, run_name, 'analysis'),
                plot_filename=f'arrhenius_{n_h}H.png',
            )
            arrhenius_records.append({
                'struct': struct_stem,
                'n_h':    n_h,
                'Ea':     arr['Ea'],
                'Ea_err': arr.get('Ea_err', 0.0),
                'D0':     arr['D0'],
                'D0_err': arr.get('D0_err', 0.0),
            })

    if arrhenius_records:
        arr_df = pd.DataFrame(arrhenius_records)
        display(arr_df)

        fig, ax = plt.subplots(figsize=(6, 4))
        for struct, grp in arr_df.groupby('struct'):
            grp = grp.sort_values('n_h')
            ax.errorbar(
                grp['n_h'], grp['Ea'],
                yerr=grp['Ea_err'],
                fmt='o-', label=struct, capsize=4,
            )
        ax.set_xlabel('Number of H atoms')
        ax.set_ylabel('Activation energy Ea (eV)')
        ax.set_title('Arrhenius Ea vs H concentration')
        ax.legend()
        fig.tight_layout()
        plt.savefig(os.path.join(RESULTS_ROOT, 'Ea_vs_nH.png'), dpi=150)
        plt.show()
        print(f'Saved: {os.path.join(RESULTS_ROOT, "Ea_vs_nH.png")}')
    else:
        print('No Arrhenius fits computed.')


In [ ]:

# ── Arrhenius overlay ─────────────────────────────────────────────────────────
# log₁₀(D) vs 1000/T for all n_H, overlaid on one axes per structure.
# Solid lines = Arrhenius fits; markers + error bars = measured D(T).

from models.diffusivity_post_processing import arrhenius_D

if 'arr_df' not in dir() or arr_df.empty:
    print('Run the Ea vs n_H cell first.')
else:
    T_fit = np.linspace(min(temperatures) - 20, max(temperatures) + 20, 200)
    inv_T_fit = 1000.0 / T_fit

    structs = arr_df['struct'].unique()
    fig, axes = plt.subplots(
        1, len(structs),
        figsize=(5 * len(structs), 4),
        sharey=True, squeeze=False,
    )
    cmap = cm.get_cmap('viridis', len(n_h_values))
    color_map = {nh: cmap(i) for i, nh in enumerate(sorted(n_h_values))}

    for col, struct in enumerate(structs):
        ax = axes[0][col]
        sub_sum = summary[summary['struct'] == struct]
        sub_arr = arr_df[arr_df['struct'] == struct]

        for _, row in sub_arr.iterrows():
            n_h = int(row['n_h'])
            color = color_map[n_h]

            # Arrhenius fit line
            D_line = arrhenius_D(T_fit, row['Ea'], row['D0'])
            ax.plot(inv_T_fit, np.log10(D_line), '-', color=color, lw=1.5)

            # Measured points  (columns: T_K, D, sigma_D, R2)
            d = sub_sum[sub_sum['n_h'] == n_h].sort_values('T_K')
            ax.errorbar(
                1000.0 / d['T_K'],
                np.log10(d['D']),
                yerr=d['sigma_D'] / (np.log(10) * d['D']),
                fmt='o', color=color, capsize=3,
                label=f'{n_h} H  Ea={row["Ea"]:.2f} eV',
            )

        ax.set_xlabel('1000 / T  (K⁻¹)')
        ax.set_title(struct)
        ax.legend(fontsize=7, title='n_H', loc='lower right')

    axes[0][0].set_ylabel('log₁₀ D  (m²/s)')
    fig.suptitle('Arrhenius Plot — H Diffusivity', fontsize=13)
    fig.tight_layout()
    plt.savefig(os.path.join(RESULTS_ROOT, 'arrhenius_overlay.png'), dpi=150)
    plt.show()
    print(f'Saved: {os.path.join(RESULTS_ROOT, "arrhenius_overlay.png")}')
